# StreamVault Exploratory Data Profiling

Investigation, not production code. It reads the source data as text so nothing is converted or
hidden prematurely, profiles every column, checks every claim in the brief against reality,
and reveals the exact rows behind the numbers.

**To use it:** *Run All* cells in order.


In [12]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

DATA = Path("../data") if Path.cwd().name == "profiling" else Path("data")
RAW = DATA / "raw"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 1000)

## 1. The candidate list, exactly as received

In [13]:
def load():
    """Read candidate lines directly from id_list.csv without dropping blank rows."""
    with open(DATA / "id_list.csv", "r", encoding="utf-8") as f:
        lines = f.read().splitlines()
    
    header = lines[0]
    data_lines = lines[1:]
    
    raw = pd.DataFrame({
        "row_number": range(1, len(data_lines) + 1),
        "raw_id": data_lines
    })
    return raw

raw = load()
print(f"Read {len(raw)} candidate lines from id_list.csv (expected 127)")
raw.head(15)

Read 127 candidate lines from id_list.csv (expected 127)


,row_number,raw_id
0,1,155
1,2,165
2,3,242
3,4,49026
4,5,8077
5,6,331
6,7,680
7,8,58
8,9,10138
9,10,673


In [14]:
# Inspect a sample raw JSON response from data/raw/ (if fetched)
sample_file = RAW / "155_details.json"
if sample_file.exists():
    with open(sample_file, "r", encoding="utf-8") as f:
        sample_data = json.load(f)
    print(f"Title: {sample_data.get('title')} | ID: {sample_data.get('id')}")
    print(f"Collection: {sample_data.get('belongs_to_collection')}")
    print(f"Genres: {sample_data.get('genres')}")
else:
    print("Raw JSON files not yet present. Run: python -m pipeline.extract")

Title: The Dark Knight | ID: 155
Collection: {'id': 263, 'name': 'The Dark Knight Collection', 'poster_path': '/ogyw5LTmL53dVxsppcy8Dlm30Fu.jpg', 'backdrop_path': '/xyhrCEdB4XRkelfVsqXeUZ6rLHi.jpg'}
Genres: [{'id': 28, 'name': 'Action'}, {'id': 53, 'name': 'Thriller'}, {'id': 80, 'name': 'Crime'}]


## 2. Every column: blanks, distinct values, commonest values

In [15]:
def columns(df: pd.DataFrame) -> pd.DataFrame:
    """One row per column: how many blanks, how many distinct values, the commonest values."""
    results = []
    for c in df.columns:
        s = df[c].astype(str).str.strip()
        blanks = int(s.isin(["", "nan", "None"]).sum())
        valid = df.loc[~s.isin(["", "nan", "None"]), c].astype(str)
        distinct = int(valid.nunique())
        commonest = ", ".join(valid.value_counts().index[:3])
        results.append({
            "column": c,
            "blank": blanks,
            "distinct": distinct,
            "commonest (not blank)": commonest
        })
    return pd.DataFrame(results).set_index("column")

print("Candidate ID Column Profile:")
columns(raw)

Candidate ID Column Profile:


,blank,distinct,commonest (not blank)
column,,,
row_number,0,127,"1, 2, 3"
raw_id,1,120,"155, 680, 278"


In [16]:
# Profile the flattened movies dataset (1 row per film grain)
movies_file = DATA / "movies.csv"
if movies_file.exists():
    df_movies = pd.read_csv(movies_file)
    print(f"Movies Table Profile: {len(df_movies)} rows, unique tmdb_id: {df_movies['tmdb_id'].is_unique}")
    display(columns(df_movies))
else:
    print("movies.csv not yet generated. Run: python -m pipeline.transform")

Movies Table Profile: 112 rows, unique tmdb_id: True


,blank,distinct,commonest (not blank)
column,,,
tmdb_id,0,112,"155, 165, 242"
imdb_id,0,112,"tt0468569, tt0096874, tt0099674"
title,0,112,"The Dark Knight, Back to the Future Part II, The Godfather Part III"
original_title,0,112,"The Dark Knight, Back to the Future Part II, The Godfather Part III"
release_date,0,112,"2008-07-16, 1989-11-22, 1990-12-25"
release_year,0,43,"2001, 2004, 2003"
status,0,1,Released
runtime,0,64,"92, 124, 136"
budget,0,60,"150000000.0, 250000000.0, 200000000.0"


## 3. The spec's claims, checked

One line per investigation question, answered with exact numbers:
- Does the list contain 127 film identifiers?
- How many duplicates and invalid formats are present?
- How many IDs return HTTP 404 from TMDb?
- What is the distribution of franchise bundles vs. standalones?
- How many titles fall before vs. after 2000 (Yaw in Finance's concern)?

In [17]:
def checks(raw: pd.DataFrame):
    """Answer the brief's questions with exact numbers."""
    s = raw["raw_id"].str.strip()
    
    # Validation tests
    is_empty = s.eq("")
    numeric_val = pd.to_numeric(s, errors="coerce")
    is_invalid_str = s.ne("") & numeric_val.isna()
    is_negative = numeric_val.le(0)
    is_valid_int = numeric_val.gt(0)
    
    valid_ids = numeric_val[is_valid_int].astype(int)
    duplicates = valid_ids.duplicated()
    unique_candidates = valid_ids.drop_duplicates()
    
    # Output summary dict
    report = {
        "Total lines in id_list.csv (excluding header)": len(raw),
        "Brief stated count": 127,
        "Blank / empty rows in file": int(is_empty.sum()),
        "Non-numeric string IDs": int(is_invalid_str.sum()),
        "Non-positive integer IDs": int(is_negative.sum()),
        "Total syntactically invalid rows": int(is_empty.sum() + is_invalid_str.sum() + is_negative.sum()),
        "Duplicate candidate IDs": int(duplicates.sum()),
        "Unique valid IDs to extract": len(unique_candidates),
    }
    
    # Add fetch log results if available
    log_file = DATA / "fetch_log.csv"
    if log_file.exists():
        log_df = pd.read_csv(log_file)
        disp = log_df["disposition"].value_counts().to_dict()
        report["Fetch log accounted total"] = len(log_df)
        report["Successfully fetched (200 OK)"] = disp.get("FETCHED", 0)
        report["Fictional IDs (HTTP 404 Not Found)"] = disp.get("FETCH_FAILED", 0)
        
    # Add movies dataset results if available
    if (DATA / "movies.csv").exists():
        m = pd.read_csv(DATA / "movies.csv")
        report["Franchise titles in catalog"] = int(m["is_franchise"].sum())
        report["Standalone titles in catalog"] = int((~m["is_franchise"]).sum())
        report["Pre-2000 titles (Finance concern)"] = int((m["release_year"] < 2000).sum())
        report["2000-onward titles (Finance concern)"] = int((m["release_year"] >= 2000).sum())
        report["Titles with zero budget"] = int((m["budget"] == 0).sum())
        report["Titles with zero revenue"] = int((m["revenue"] == 0).sum())
        
    return report

pd.DataFrame({"StreamVault Candidate Audit": checks(raw)})

,StreamVault Candidate Audit
Total lines in id_list.csv (excluding header),127
Brief stated count,127
Blank / empty rows in file,1
Non-numeric string IDs,2
Non-positive integer IDs,1
Total syntactically invalid rows,4
Duplicate candidate IDs,6
Unique valid IDs to extract,117
Fetch log accounted total,127
Successfully fetched (200 OK),112


## 4. The rows behind the numbers

Counts tell you *how many*. Rows tell you *why*.

In [18]:
# Repeated candidate IDs, side by side
s = raw["raw_id"].str.strip()
numeric_ids = pd.to_numeric(s, errors="coerce")
dup_mask = numeric_ids.duplicated(keep=False) & numeric_ids.notna()

print("Duplicate rows in id_list.csv (side by side):")
raw[dup_mask].sort_values("raw_id")

Duplicate rows in id_list.csv (side by side):


,row_number,raw_id
0,1,155
89,90,155
53,54,1891
88,89,1891
25,26,278
58,59,278
78,79,550
92,93,550
38,39,603
41,42,603


In [19]:
# Syntactically invalid candidate rows
invalid_mask = s.eq("") | numeric_ids.isna() | numeric_ids.le(0)

print("Syntactically invalid rows:")
raw[invalid_mask]

Syntactically invalid rows:


,row_number,raw_id
28,29,-1
67,68,
93,94,tt0468569
109,110,N/A


In [20]:
# Non-existent candidate IDs (HTTP 404 Not Found from TMDb)
log_file = DATA / "fetch_log.csv"
if log_file.exists():
    log_df = pd.read_csv(log_file)
    failed_404s = log_df[log_df["disposition"] == "FETCH_FAILED"]
    print("Fictional / non-existent IDs in candidate list (HTTP 404):")
    display(failed_404s[["row", "raw", "reason"]])
else:
    print("fetch_log.csv not yet generated.")

Fictional / non-existent IDs in candidate list (HTTP 404):


,row,raw,reason
16,17,8471293,ID not found in TMDb (details 404)
33,34,9412887,ID not found in TMDb (details 404)
71,72,8471294,ID not found in TMDb (details 404)
91,92,9412888,ID not found in TMDb (details 404)
97,98,7733901,ID not found in TMDb (details 404)


In [21]:
# Top Franchise Collections in catalog by total box office
if (DATA / "movies.csv").exists():
    m = pd.read_csv(DATA / "movies.csv")
    collections = (
        m[m["is_franchise"]]
        .groupby(["collection_id", "collection_name"])
        .agg(
            titles=("tmdb_id", "count"),
            total_revenue=("revenue", "sum"),
            mean_revenue=("revenue", "mean"),
            mean_vote=("vote_average", "mean")
        )
        .reset_index()
        .sort_values(by="total_revenue", ascending=False)
    )
    
    top_collections = collections.head(8).copy()
    top_collections["total_revenue"] = top_collections["total_revenue"].apply(lambda x: f"${x:,.0f}")
    top_collections["mean_revenue"] = top_collections["mean_revenue"].apply(lambda x: f"${x:,.0f}")
    top_collections["mean_vote"] = top_collections["mean_vote"].apply(lambda x: f"{x:.2f}")
    
    print("Top Franchise Collections in candidate catalog:")
    display(top_collections)
else:
    print("movies.csv not yet generated.")

Top Franchise Collections in candidate catalog:


,collection_id,collection_name,titles,total_revenue,mean_revenue,mean_vote
23,86311.0,The Avengers Collection,4,"$7,919,284,026","$1,979,821,006",7.97
10,1241.0,Harry Potter Collection,8,"$7,759,777,709","$969,972,214",7.83
6,295.0,Pirates of the Caribbean Collection,5,"$4,525,525,744","$905,105,149",7.15
0,10.0,Star Wars Collection,6,"$4,432,411,744","$738,735,291",7.53
21,10194.0,Toy Story Collection,4,"$3,039,690,868","$759,922,717",7.72
12,2150.0,Shrek Collection,4,"$2,983,130,036","$745,782,509",6.96
26,121938.0,The Hobbit Collection,3,"$2,935,523,356","$978,507,785",7.44
2,119.0,The Lord of the Rings Collection,3,"$2,916,544,743","$972,181,581",8.46


In [22]:
# Dual-Window Comparison: All Years vs. 2000-Onward (Yaw in Finance)
if (DATA / "movies.csv").exists():
    m = pd.read_csv(DATA / "movies.csv")
    m["roi"] = (m["revenue"] - m["budget"]) / m["budget"]
    
    def window_kpis(df, window_label):
        agg = df.groupby("is_franchise").agg(
            count=("tmdb_id", "count"),
            mean_revenue=("revenue", "mean"),
            median_revenue=("revenue", "median"),
            mean_roi=("roi", "mean"),
            median_roi=("roi", "median"),
            mean_vote=("vote_average", "mean")
        ).reset_index()
        agg["type"] = np.where(agg["is_franchise"], "Franchise Bundle", "Standalone Hit")
        agg["window"] = window_label
        return agg[["window", "type", "count", "mean_revenue", "median_revenue", "mean_roi", "median_roi", "mean_vote"]]
    
    kpi_all = window_kpis(m, "All Years")
    kpi_post_2000 = window_kpis(m[m["release_year"] >= 2000], "2000-Onward")
    comp = pd.concat([kpi_all, kpi_post_2000], ignore_index=True)
    
    display_comp = comp.copy()
    display_comp["mean_revenue"] = display_comp["mean_revenue"].apply(lambda x: f"${x:,.0f}")
    display_comp["median_revenue"] = display_comp["median_revenue"].apply(lambda x: f"${x:,.0f}")
    display_comp["mean_roi"] = display_comp["mean_roi"].apply(lambda x: f"{x:.2f}x")
    display_comp["median_roi"] = display_comp["median_roi"].apply(lambda x: f"{x:.2f}x")
    display_comp["mean_vote"] = display_comp["mean_vote"].apply(lambda x: f"{x:.2f}")
    
    print("Two-Window Comparison of Franchise Bundles vs. Standalone Hits:")
    display(display_comp)
else:
    print("movies.csv not yet generated.")

Two-Window Comparison of Franchise Bundles vs. Standalone Hits:

,window,type,count,mean_revenue,median_revenue,mean_roi,median_roi,mean_vote
0,All Years,Standalone Hit,16,"$270,271,171","$161,802,536",7.28x,2.84x,8.37
1,All Years,Franchise Bundle,96,"$654,556,698","$578,937,111",7.54x,4.62x,7.46
2,2000-Onward,Standalone Hit,8,"$353,808,227","$283,195,048",3.90x,2.84x,8.26
3,2000-Onward,Franchise Bundle,63,"$792,986,415","$786,636,033",4.41x,4.05x,7.33
